# 🧹 Data Wrangling
## Twitter Sentiment Analysis — Mental Health Topic

**Pipeline:**
1. Gathering Data
2. Assessing Data
3. Cleaning Data
4. Business Questions

> **Output:** `datasets/processed/twitter_clean.csv`, `top_hashtags.json`, `ab_results.json`, `assessment.json`

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import ast
import json
import os
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

In [2]:
# ── Konfigurasi Path ──────────────────────────────────────────────────────────
RAW_PATH       = Path('../datasets/raw/Twitter_Analysis.xlsx')
PROCESSED_DIR  = Path('../datasets/processed')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Mapping label numerik → teks
LABEL_MAP = {2: 'Positive', 1: 'Neutral', 0: 'Negative'}

---
## 1. Gathering Data

In [3]:
# Dataset: tweet bertema Mental Health dengan label sentimen manual
# Label: Bagus (2=Positive) | Cukup (1=Neutral) | Buruk (0=Negative)

try:
    df_raw = pd.read_excel(RAW_PATH)
    print(f'✅ Dataset dimuat: {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom')
except FileNotFoundError:
    raise SystemExit(
        f'[ERROR] File tidak ditemukan: {RAW_PATH}\n'
        'Pastikan file Excel berada di datasets/raw/'
    )

✅ Dataset dimuat: 12,492 baris × 6 kolom


---
## 2. Assessing Data

In [4]:
# 2.1 Dimensi & Tipe Data
print(f'Shape  : {df_raw.shape}')
print(f'Kolom  : {df_raw.columns.tolist()}\n')
df_raw.dtypes

Shape  : (12492, 6)
Kolom  : ['text', 'hashtags', 'labels', 'label_text', 'text_length', 'hashtag_count']



text             object
hashtags         object
labels            int64
label_text       object
text_length       int64
hashtag_count     int64
dtype: object

In [5]:
# 2.2 Missing Values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

,Missing Count,Missing %
text,0,0.0
hashtags,0,0.0
labels,0,0.0
label_text,0,0.0
text_length,0,0.0
hashtag_count,0,0.0


In [6]:
# 2.3 Duplikat
n_dup_full = df_raw.duplicated().sum()
n_dup_text = df_raw.duplicated(subset=['text']).sum()

print(f'Duplikat baris penuh  : {n_dup_full:,} ({n_dup_full / len(df_raw) * 100:.1f}%)')
print(f"Duplikat kolom 'text' : {n_dup_text:,} ({n_dup_text / len(df_raw) * 100:.1f}%)")

Duplikat baris penuh  : 3,907 (31.3%)
Duplikat kolom 'text' : 4,053 (32.4%)


In [7]:
# 2.4 Distribusi Kelas & Statistik Deskriptif
print('Distribusi label (raw):')
print(df_raw['label_text'].value_counts().to_string())
print()
df_raw[['text_length', 'hashtag_count']].describe().round(2)

Distribusi label (raw):
label_text
Bagus    4164
Cukup    4164
Buruk    4164



,text_length,hashtag_count
count,12492.00,12492.00
mean,180.81,4.32
std,73.88,4.96
min,11.00,0.00
25%,121.00,1.00
50%,193.00,3.00
75%,240.00,6.00
max,1019.00,30.00


In [8]:
# Simpan laporan assessment ke JSON
dist_raw = df_raw['label_text'].value_counts()

assessment_report = {
    'total_rows_raw'    : int(len(df_raw)),
    'total_cols'        : int(len(df_raw.columns)),
    'missing_values'    : {k: int(v) for k, v in missing.items()},
    'duplicate_rows'    : int(n_dup_full),
    'duplicate_text'    : int(n_dup_text),
    'class_distribution': {k: int(v) for k, v in dist_raw.items()},
    'text_length_mean'  : float(df_raw['text_length'].mean()),
    'text_length_max'   : float(df_raw['text_length'].max()),
    'hashtag_count_mean': float(df_raw['hashtag_count'].mean()),
}

out_path = PROCESSED_DIR / 'assessment.json'
with open(out_path, 'w') as f:
    json.dump(assessment_report, f, indent=2)

print(f'✅ Laporan assessment disimpan → {out_path}')

✅ Laporan assessment disimpan → ..\datasets\processed\assessment.json


---
## 3. Cleaning Data

In [9]:
# 3.1 Deduplikasi berdasarkan kolom 'text'
df = df_raw.drop_duplicates(subset=['text']).copy().reset_index(drop=True)

removed = len(df_raw) - len(df)
print(f'Sebelum : {len(df_raw):,} baris')
print(f'Dihapus : {removed:,} duplikat ({removed / len(df_raw) * 100:.1f}%)')
print(f'Sesudah : {len(df):,} baris')

Sebelum : 12,492 baris
Dihapus : 4,053 duplikat (32.4%)
Sesudah : 8,439 baris


In [10]:
# 3.2 Pembersihan Teks: hapus URL dan normalisasi whitespace
def clean_text(text: str) -> str:
    """Hapus URL dan normalisasi whitespace dari teks tweet."""
    text = str(text).strip()
    text = re.sub(r'http\S+|www\S+', '', text)  # hapus URL
    text = re.sub(r'\s+', ' ', text)            # normalisasi spasi
    return text.strip()


df['text_clean'] = df['text'].apply(clean_text)

n_with_url = df['text'].str.contains(r'http\S+|www\S+', regex=True).sum()
print(f'Tweet dengan URL (sebelum dibersihkan): {n_with_url:,}')
print('✅ URL dan whitespace berlebih berhasil dibersihkan')

Tweet dengan URL (sebelum dibersihkan): 0
✅ URL dan whitespace berlebih berhasil dibersihkan


In [11]:
# 3.3 Parsing Hashtag: dari string repr Python list → Python list sesungguhnya
def parse_hashtags(raw: str) -> list[str]:
    """
    Konversi string representasi list ke Python list.

    Input  : "['Stress', 'anxiety']"
    Output : ['stress', 'anxiety']
    """
    try:
        items = ast.literal_eval(str(raw))
        return [
            re.sub(r"['\"\s]", '', str(item)).lower().strip()
            for item in items
            if len(str(item).strip()) > 1
        ]
    except Exception:
        return []


df['hashtag_list'] = df['hashtags'].apply(parse_hashtags)

print('Contoh hasil parsing hashtag:')
for i in range(3):
    print(f'  [{i}] {df["hashtag_list"].iloc[i]}')

Contoh hasil parsing hashtag:
  [0] ['stress', 'infertility', 'lifestylefactors', 'fertility', 'uscfertility']
  [1] ['stress']
  [2] ['churchmentalhealth', 'mentalhealthmatters', 'cmhs21']


In [12]:
# 3.4 Mapping Sentimen
df['sentiment'] = df['labels'].map(LABEL_MAP)

print('✅ Kolom sentiment ditambahkan:')
print(df['sentiment'].value_counts().to_string())


✅ Kolom sentiment ditambahkan:
sentiment
Negative    4130
Neutral     2804
Positive    1505


In [13]:
# 3.5 Validasi Pasca-Cleaning
print(f'Shape akhir      : {df.shape}')
print(f'Missing values   : {df.isnull().sum().sum()}')
print(f'Duplikat tersisa : {df.duplicated(subset=["text"]).sum()}')
print(f'Kolom final      : {df.columns.tolist()}')

Shape akhir      : (8439, 9)
Missing values   : 0
Duplikat tersisa : 0
Kolom final      : ['text', 'hashtags', 'labels', 'label_text', 'text_length', 'hashtag_count', 'text_clean', 'hashtag_list', 'sentiment']


In [14]:
# Simpan dataset bersih (tanpa kolom hashtag_list yang tidak serializable)
df_to_save = df.drop(columns=['hashtag_list'])
clean_path = PROCESSED_DIR / 'twitter_clean.csv'
df_to_save.to_csv(clean_path, index=False)

print(f'✅ Dataset bersih disimpan → {clean_path}')
print(f'   {len(df_to_save):,} baris × {len(df_to_save.columns)} kolom')
print(f'   Kolom: {df_to_save.columns.tolist()}')


✅ Dataset bersih disimpan → ..\datasets\processed\twitter_clean.csv
   8,439 baris × 8 kolom
   Kolom: ['text', 'hashtags', 'labels', 'label_text', 'text_length', 'hashtag_count', 'text_clean', 'sentiment']


In [15]:
# Simpan top 20 hashtag per sentimen
top_hashtags: dict[str, list] = {}

for sentiment in ['Positive', 'Neutral', 'Negative']:
    all_tags = [
        tag
        for tag_list in df[df['sentiment'] == sentiment]['hashtag_list']
        for tag in tag_list
    ]
    top_hashtags[sentiment] = [
        [tag, int(count)]
        for tag, count in Counter(all_tags).most_common(20)
    ]

hashtag_path = PROCESSED_DIR / 'top_hashtags.json'
with open(hashtag_path, 'w') as f:
    json.dump(top_hashtags, f, indent=2)

print(f'✅ Top hashtags disimpan → {hashtag_path}')
for sentiment in ['Positive', 'Neutral', 'Negative']:
    print(f'   {sentiment:<10} top-3: {top_hashtags[sentiment][:3]}')

✅ Top hashtags disimpan → ..\datasets\processed\top_hashtags.json
   Positive   top-3: [['mentalhealth', 415], ['stress', 331], ['mentalhealthmatters', 202]]
   Neutral    top-3: [['mentalhealth', 771], ['stress', 652], ['mentalhealthmatters', 406]]
   Negative   top-3: [['happy', 1012], ['excited', 757], ['mentalhealth', 558]]


---
## 4. Business Questions

| # | Pertanyaan | Metode |
|---|-----------|--------|
| BQ1 | Apakah distribusi sentimen seimbang? | Descriptive Stats |
| BQ2 | Hashtag apa yang paling dominan per sentimen? | Word Frequency |
| BQ3 | Apakah jumlah hashtag berbeda antar sentimen? | Mann-Whitney U |
| BQ4 | Apakah panjang teks berbeda antar sentimen? | Kruskal-Wallis |
| BQ5 | Adakah pola ironi dalam penggunaan kata? | Analisis Hashtag |

> Jawaban detail di notebook `02_eda.ipynb` dan `03_ab_testing.ipynb`